# ContextCheck: Colab GPU training

This notebook runs the project's real SNLI preprocessing, baseline, and DistilBERT training scripts. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another GPU). Upload a ZIP of the project source (exclude model weights, caches, and downloaded data). Set `TRAIN_LIMIT` to `None` for full training or a smaller integer for a repeatable pilot run. Test examples are reserved for final evaluation.

In [ ]:
from google.colab import files
from pathlib import Path
import os
import zipfile

uploaded = files.upload()  # Select a ZIP containing app.py, requirements.txt, and src/.
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
project_root = Path('/content/contextcheck')
project_root.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(project_root)
candidates = [path for path in project_root.rglob('requirements.txt') if (path.parent / 'src').is_dir()]
if not candidates:
    raise FileNotFoundError('The uploaded ZIP must contain the ContextCheck project source.')
os.chdir(candidates[0].parent)
print('Project directory:', Path.cwd())

In [ ]:
%pip install -q -r requirements.txt

## Inspect and preprocess SNLI

The loader prints the feature structure and row counts before and after removing unlabeled or unusable examples, while preserving the original split boundaries.

In [ ]:
from src.dataset_loader import load_snli
snli, preprocessing_stats = load_snli()
preprocessing_stats

In [ ]:
# None means use all training examples; e.g. 20000 is a deterministic pilot subset.
TRAIN_LIMIT = 20000
EPOCHS = 2

import subprocess, sys
common = ['--max-train-samples', str(TRAIN_LIMIT)] if TRAIN_LIMIT else []
subprocess.run([sys.executable, '-m', 'src.train_baseline', *common], check=True)
subprocess.run([
    sys.executable, '-m', 'src.train_transformer',
    '--epochs', str(EPOCHS), '--batch-size', '8',
    '--gradient-accumulation-steps', '2', *common
], check=True)

## Save artifacts to Google Drive

Model weights can be large. This optional cell copies the fine-tuned model and measured reports to Drive so the Streamlit app can load them locally later. Do not commit the weights to Git.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
destination = Path('/content/drive/MyDrive/ContextCheck')
destination.mkdir(parents=True, exist_ok=True)
for folder in ('models/contextcheck_distilbert', 'models/baseline', 'artifacts'):
    source = Path(folder)
    if source.exists():
        target = destination / source
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(source, target, dirs_exist_ok=True)
print('Saved trained artifacts under', destination)